# 🚀 Topic 05: Spark Optimization, Shuffling & Broadcast Joins

## 1. The Shuffling Problem
Shuffling is the process of redistributing data across executors over the network.
- **Cost:** High disk I/O, heavy network serialization, memory pressure.
- **Goal:** Minimize shuffles wherever possible.

---

## 2. Join Strategies
1. **Sort-Merge Join (Default for Large Tables):** Both tables are shuffled by join key, sorted, and merged.
2. **Broadcast Hash Join (Map-Side Join):** Small table (<10MB default) is broadcasted to all executor nodes. **Zero Network Shuffle for the large table!**

---

## 3. Hands-on: Broadcast Join Optimization & Salting Skewed Keys


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.master("local[*]").appName("Spark_Optimization_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Create Large DataFrame (Fact Table)
large_fact = spark.range(1, 100000).withColumn("store_id", (F.col("id") % 5).cast("int"))

# Create Small DataFrame (Dimension Table)
small_dim = spark.createDataFrame([
    (0, "Downtown Store"),
    (1, "Suburban Mall Store"),
    (2, "Airport Kiosk"),
    (3, "Online Flagship"),
    (4, "Highway Outlet")
], ["store_id", "store_name"])

# 1. Standard Join (May trigger shuffle)
df_standard_join = large_fact.join(small_dim, on="store_id")

# 2. Optimized Broadcast Join
df_broadcast_join = large_fact.join(F.broadcast(small_dim), on="store_id")

print("🔍 Physical Execution Plan for Broadcast Join:")
# Notice 'BroadcastHashJoin' in execution plan
df_broadcast_join.explain(True)
